In [1]:
import pandas as pd
import re
import os
import sys
sys.path.append("stable-baselines3")
sys.path.append("..")
sys.path.append("强化学习")
import gymnasium as gym
from 强化学习 import load_datas,LivestockEnvConfig
from gymnasium.envs.registration import register
# from livestockEnvV2 import load_datas,LivestockEnvConfig

register(
    id='LivestockEnv-v2',
    entry_point='livestockEnvV2:LivestockEnv',
)
country = 'cn'
config = LivestockEnvConfig(country, 
                            Reward_priority=[4, 4, 3, 2, 1], 
                            thresholds=[0, 31, 0], 
                            mobility_ratio=0.25,
                            max_steps=5000)
env = gym.make('LivestockEnv-v2', config=config)  


In [2]:
country_mapping = {
'usa': '美国',
'br': '巴西',
'us': '欧盟',
'cn': '中国',
'aus': '澳大利亚'
}

In [3]:
df = pd.read_excel(rf'../results/{country}//PPO1.xlsx', index_col=0)

ID_move_in, ID_move_out, Move_in, Move_out, Target_move_in, Coef_move_in, Target_move_out, Coef_move_out = load_datas(country)
if not os.path.exists(f"../results/{country}/"):
    os.makedirs(f"../results/{country}/")

### 合并移动方案

In [4]:
import torch
import re

def string_to_tensor(string):
    # 使用正则表达式提取数字部分
    numbers = re.findall(r'\d+', string)
    # 将数字转换为列表
    numbers = [int(num) for num in numbers]
    # 创建 tensor
    tensor = torch.tensor(numbers)
    return tensor


In [5]:
df["amount"] = df["amount"].apply(lambda x : string_to_tensor(x))

df[Move_out.columns] = 0

for i in range(len(df)):
    for a in range(env.action_len):
        df.iloc[i, a-env.action_len] = df.loc[i, "amount"][a].item()

df.drop(["amount", "action_mask.sum()"], axis=1, inplace=True)
merged_data = df.groupby(['move_in_idx', 'move_out_idx']).agg(sum).reset_index()
merged_data.to_excel(f'../results/{country}/PPO_concated.xlsx', index=False)
merged_data

/home/yanisy/.local/lib/python3.11/site-packages/gymnasium/core.py:311: UserWarning: WARN: env.action_len to get variables from other wrappers is deprecated and will be removed in v1.0, to get this variable you can do `env.unwrapped.action_len` for environment variables or `env.get_wrapper_attr('action_len')` that will search the reminding wrappers.
  logger.warn(
/tmp/ipykernel_3839276/4143208668.py:10: FutureWarning: The provided callable <built-in function sum> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  merged_data = df.groupby(['move_in_idx', 'move_out_idx']).agg(sum).reset_index()


,move_in_idx,move_out_idx,reward,num_dairyhead,num_beefcattle,num_pig,num_lay,num_poultry,num_sheep
0,0,653,16.987267,0,40,821,1302,3217,0
1,0,1461,10.728446,467,1767,47264,782074,1392624,5980
2,0,1616,45.986584,8073,30392,13570,7492,18284,14319
3,0,1894,20.663154,8,146,9107,105106,256512,2973
4,1,845,22.895979,39,1542,94199,746352,1821525,3124
...,...,...,...,...,...,...,...,...,...
1580,876,1660,22.803551,678,2973,39741,84905,207136,78300
1581,877,671,14.954313,9619,7278,9742,37054,90679,9484
1582,878,1988,6.680768,0,0,0,0,688691,0
1583,878,1998,14.966687,187,126,64370,210860,514627,14516


### 生成移动后移出城市剩余数量

In [6]:
N_demand_Move_out = Target_move_out['N_demand']
ammonia_density_Move_out = Target_move_out['ammonia_density']
livestock_PB_Move_out = Target_move_out['livestock_PB']
Move_out_tensor_Coef_N_demand = torch.tensor(Coef_move_out['N_demand'].values)
Move_out_tensor_Coef_ammonia_density = torch.tensor(Coef_move_out['ammonia_density'].values)
Move_out_tensor_Coef_livestock_PB = torch.tensor(Coef_move_out['livestock_PB'].values)

In [7]:
import torch
merged_data = pd.read_excel(f'../results/{country}/PPO_concated.xlsx')
Move_out_copy = Move_out.copy()
for idx, line in merged_data.iterrows():
    out_idx = int(line["move_out_idx"])
    Move_out_copy.iloc[out_idx, :] -= line.iloc[-env.action_len:]
    amounts = torch.tensor(line.iloc[-env.action_len:].values)
    N_demand_Move_out[out_idx] -= (amounts.double() @ Move_out_tensor_Coef_N_demand[out_idx, :]).item()
    ammonia_density_Move_out[out_idx] -= (amounts.double() @ Move_out_tensor_Coef_ammonia_density[out_idx, :]).item()
    livestock_PB_Move_out[out_idx] -= (amounts.double() @ Move_out_tensor_Coef_livestock_PB[out_idx, :]).item()
pd.concat([ID_move_out,Move_out_copy,N_demand_Move_out,ammonia_density_Move_out,livestock_PB_Move_out], axis=1).to_excel(f"../results/{country}/move_out_result1.xlsx", index=False)
# pd.concat([ID_move_out,Move_out_copy], axis=1).to_excel(f"../results/{country}/move_out_result1.xlsx", index=False)


/home/yanisy/.local/lib/python3.11/site-packages/gymnasium/core.py:311: UserWarning: WARN: env.action_len to get variables from other wrappers is deprecated and will be removed in v1.0, to get this variable you can do `env.unwrapped.action_len` for environment variables or `env.get_wrapper_attr('action_len')` that will search the reminding wrappers.
  logger.warn(


### 生成移动后移入城市剩余数量

In [8]:
N_demand_Move_in = Target_move_in['N_demand']
ammonia_density_Move_in = Target_move_in['ammonia_density']
livestock_PB_Move_in = Target_move_in['livestock_PB']
Move_in_tensor_Coef_N_demand = torch.tensor(Coef_move_in['N_demand'].values)
Move_in_tensor_Coef_ammonia_density = torch.tensor(Coef_move_in['ammonia_density'].values)
Move_in_tensor_Coef_livestock_PB = torch.tensor(Coef_move_in['livestock_PB'].values)

In [9]:
Move_in_copy = Move_in.copy()

for idx, line in merged_data.iterrows():
    out_idx = int(line["move_out_idx"])
    in_idx = int(line["move_in_idx"])
    Move_in_copy.iloc[in_idx, :] += line.iloc[-env.action_len:]
    amounts = torch.tensor(line.iloc[-env.action_len:].values)
    N_demand_Move_in[in_idx] += (amounts.double() @ Move_in_tensor_Coef_N_demand[in_idx, :]).item()
    ammonia_density_Move_in[in_idx] += (amounts.double() @ Move_in_tensor_Coef_ammonia_density[in_idx, :]).item()
    livestock_PB_Move_in[in_idx] += (amounts.double() @ Move_in_tensor_Coef_livestock_PB[in_idx, :]).item()
pd.concat([ID_move_in,Move_in_copy,N_demand_Move_in,ammonia_density_Move_in,livestock_PB_Move_in], axis=1).to_excel(f"../results/{country}/move_in_result1.xlsx", index=False)
# Move_in_copy[[*Move_in_copy.columns[:env.k],"氨排放差距","承载力差距",*Move_out.columns[env.k:]]].to_excel(f"../results/{country}/v4/move_in_result1.xlsx", index=False)


/home/yanisy/.local/lib/python3.11/site-packages/gymnasium/core.py:311: UserWarning: WARN: env.action_len to get variables from other wrappers is deprecated and will be removed in v1.0, to get this variable you can do `env.unwrapped.action_len` for environment variables or `env.get_wrapper_attr('action_len')` that will search the reminding wrappers.
  logger.warn(
